# Camada Silver

Objetivo: transformar o dado bruto do schema `bronze` em tabelas limpas, tipadas e com um grão claro, gravadas no schema `silver`.

| Tabela | Uma linha é | Chave |
|---|---|---|
| `silver.cartas` | uma carta do catálogo (Pokémon, Treinador ou Energia) | `card_id` |
| `silver.movimentos` | um ataque, habilidade ou regra Tera de um Pokémon | `movimento_id` (ligada a `cartas` por `card_id`) |
| `silver.deck_exemplo` | uma cópia de carta dentro do deck exemplo de 60 cartas | `posicao` (ligada a `cartas` por `card_id`) |
| `silver.qualidade` | uma regra de qualidade com o número de violações | `regra` |

As funções ficam em `pipeline/silver.py` e são testadas em `tests/test_silver.py` com dados inventados.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd

from pipeline import config
from pipeline.db import conectar, ler_tabela, listar_tabelas, salvar_tabela
from pipeline.silver import checar_qualidade, construir_cartas, construir_deck, construir_movimentos

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 50)

conexao = conectar(config.BANCO)
bronze_en = ler_tabela(conexao, "bronze", "cartas_en")
bronze_jp = ler_tabela(conexao, "bronze", "cartas_jp")
bronze_deck = ler_tabela(conexao, "bronze", "deck_exemplo")

## 1. Decisões de tratamento e o motivo de cada uma

| # | Decisão | Motivo |
|---|---|---|
| 1 | `n/a` e campo vazio viram nulo | São as duas formas de "não se aplica" no bruto. Tratadas igual, as contagens e médias ignoram esses campos. |
| 2 | Separar o bruto em `cartas` e `movimentos` | O CSV repete os dados da carta em cada ataque. Sem separar, um Pokémon com dois ataques contaria duas vezes no HP médio. |
| 3 | Dados da carta pegos com o primeiro valor não nulo do `card_id` | Conferido na exploração: HP, tipo, fraqueza e demais campos são iguais em todas as linhas da mesma carta. A coluna `Species` aparece só em uma das linhas em 3 cartas. |
| 4 | Símbolos `{G}`, `{R}`... traduzidos para Grama, Fogo... | Facilita a leitura dos indicadores. O texto original fica em `custo_original` e `dano_original` para conferência. |
| 5 | `●` conta como energia incolor no custo | Na carta impressa é o símbolo da energia que aceita qualquer tipo. |
| 6 | `No cost` vira custo 0 | São 5 ataques que não exigem energia. |
| 7 | Dano separado em `dano_base` e `modificador_dano` | `70×` (multiplicador) e `120-` (redução) não são dano fixo. Os indicadores de eficiência usam só dano fixo para não superestimar ataques variáveis. |
| 8 | Prefixos `[Ability]` e `[Tera]` viram `tipo_movimento` | Habilidade não é ataque e `[Tera]` é uma regra da carta. Ficam fora das métricas de dano. |
| 9 | Recuo nulo em Pokémon vira 0 | A carta impressa sem símbolo de recuo recua de graça. Em Treinador e Energia o recuo continua nulo, porque não se aplica. |
| 10 | `premios_cedidos` = 3 para Mega ex, 2 para ex e 1 para os demais Pokémon | É a regra do jogo e é o custo de risco usado nos indicadores de HP por prêmio. |
| 11 | Nome japonês vem de um JOIN com o arquivo JP por `card_id` | Os dois arquivos descrevem as mesmas 1431 cartas. O JOIN é validado como um para um. |
| 12 | Expansão ausente (8 cartas) fica nula, sem preencher | O arquivo JP tem a sigla japonesa (`expansao_jp`), mas ela não corresponde à sigla em inglês. Preencher seria inventar. |
| 13 | Fraqueza ausente em Pokémon fica nula | Não é falha: Pokémon do tipo Dragão não têm fraqueza nesta geração do jogo, e isso vira um indicador. |

## 2. Construção das tabelas

In [2]:
cartas = construir_cartas(bronze_en, bronze_jp)
movimentos = construir_movimentos(bronze_en)
deck = construir_deck(bronze_deck, cartas)

print("silver.cartas:", cartas.shape)
print("silver.movimentos:", movimentos.shape)
print("silver.deck_exemplo:", deck.shape)

silver.cartas: (1431, 21)
silver.movimentos: (2038, 14)
silver.deck_exemplo: (60, 2)


In [3]:
cartas.head()

,card_id,nome,nome_jp,supertipo,subtipo,regra,premios_cedidos,tipo,hp,fraqueza,resistencia,recuo,evolui_de,categoria,grupo_categoria,especie,expansao,numero_colecao,expansao_jp,numero_colecao_jp,texto_carta
0,1,Basic {G} Energy,基本【草】エネルギー,Energia,Energia Básica,Comum,<NA>,Grama,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,SVE,1,<NA>,GRA,<NA>
1,2,Basic {R} Energy,基本【炎】エネルギー,Energia,Energia Básica,Comum,<NA>,Fogo,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,SVE,2,<NA>,FIR,<NA>
2,3,Basic {W} Energy,基本【水】エネルギー,Energia,Energia Básica,Comum,<NA>,Água,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,SVE,3,<NA>,WAT,<NA>
3,4,Basic {L} Energy,基本【雷】エネルギー,Energia,Energia Básica,Comum,<NA>,Elétrico,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,SVE,4,<NA>,LIG,<NA>
4,5,Basic {P} Energy,基本【超】エネルギー,Energia,Energia Básica,Comum,<NA>,Psíquico,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,SVE,5,<NA>,PSY,<NA>


In [4]:
movimentos.head()

,movimento_id,card_id,ordem,tipo_movimento,nome_movimento,custo_original,custo_total,custo_incolor,custo_colorido,dano_original,dano_base,modificador_dano,tem_efeito,efeito
0,21_1,21,1,Ataque,Nab ’n’ Dash,●,1,1,0,<NA>,<NA>,Sem dano,True,Search your deck for a number of cards up to t...
1,21_2,21,2,Ataque,High Jump Kick,{D}●●,3,2,1,100,100,Fixo,False,<NA>
2,22_1,22,1,Ataque,Push Down,{F},1,0,1,10,10,Fixo,True,Switch out your opponent’s Active Pokémon to t...
3,23_1,23,1,Ataque,Ram,{F}{F},2,0,2,60,60,Fixo,False,<NA>
4,23_2,23,2,Ataque,Super Sandstorm,{F}{F}●,3,1,2,150,150,Fixo,True,This attack also does 40 damage to each Benche...


## 3. Conferências da transformação

Volume: o número de cartas tem que ser igual ao de `Card ID` distintos no bruto, e o de movimentos igual ao de linhas com nome de movimento preenchido.

In [5]:
linhas_com_movimento = (~bronze_en["Move Name"].str.strip().isin(["", "n/a"])).sum()
assert len(cartas) == bronze_en["Card ID"].nunique()
assert len(movimentos) == linhas_com_movimento
print("Volumes conferem com o bronze")

Volumes conferem com o bronze


In [6]:
pd.DataFrame({
    "supertipo": cartas["supertipo"].value_counts(),
}).join(cartas.groupby("supertipo")["subtipo"].unique().rename("subtipos"))

,supertipo,subtipos
supertipo,,
Pokémon,1196,"[Estágio 1, Básico, Estágio 2]"
Treinador,210,"[Item, Ferramenta, Apoiador, Estádio]"
Energia,25,"[Energia Básica, Energia Especial]"


In [7]:
movimentos.groupby(["tipo_movimento", "modificador_dano"]).size().rename("qtd").reset_index()

,tipo_movimento,modificador_dano,qtd
0,Ataque,Fixo,1323
1,Ataque,Multiplicador,121
2,Ataque,Redução,3
3,Ataque,Sem dano,308
4,Habilidade,Sem dano,251
5,Regra Tera,Sem dano,32


## 4. Regras de qualidade

In [8]:
qualidade = checar_qualidade(cartas, movimentos)
assert qualidade["violacoes"].sum() == 0, "Há violações de qualidade"
qualidade

,regra,violacoes
0,card_id repetido em silver.cartas,0
1,movimento sem carta correspondente,0
2,carta sem nome japonês (JOIN EN x JP),0
3,Pokémon sem HP,0
4,Pokémon sem tipo,0
5,Pokémon sem nenhum movimento,0
6,ataque sem custo,0
7,HP fora do intervalo de 30 a 400,0


Casos conhecidos que não são violação, conforme as decisões 12 e 13:

In [9]:
pokemon = cartas[cartas["supertipo"] == "Pokémon"]
print("Cartas sem expansão:", cartas["expansao"].isna().sum())
print("Pokémon sem fraqueza por tipo:")
print(pokemon[pokemon["fraqueza"].isna()]["tipo"].value_counts())

Cartas sem expansão: 8
Pokémon sem fraqueza por tipo:
tipo
Dragão    39
Name: count, dtype: Int64


## 5. Gravação no schema silver

In [10]:
salvar_tabela(conexao, "silver", "cartas", cartas)
salvar_tabela(conexao, "silver", "movimentos", movimentos)
salvar_tabela(conexao, "silver", "deck_exemplo", deck)
salvar_tabela(conexao, "silver", "qualidade", qualidade)
listar_tabelas(conexao, "silver")

['cartas', 'deck_exemplo', 'movimentos', 'qualidade']

Conferência do JOIN no banco: o deck exemplo ligado às cartas pelo `card_id`.

In [11]:
conexao.execute("""
    SELECT c.nome, c.nome_jp, c.supertipo, COUNT(*) AS copias
    FROM silver.deck_exemplo AS d
    JOIN silver.cartas AS c ON c.card_id = d.card_id
    GROUP BY ALL
    ORDER BY copias DESC
""").df()

,nome,nome_jp,supertipo,copias
0,Basic {W} Energy,基本【水】エネルギー,Energia,35
1,Waitress,ウエートレス,Treinador,4
2,Snover,ユキカブリ,Pokémon,4
3,Mega Abomasnow ex,メガユキノオーex,Pokémon,4
4,Lillie's Determination,リーリエの決心,Treinador,4
5,Mega Signal,メガシグナル,Treinador,4
6,Cyrano,シアノ,Treinador,2
7,Kyogre,カイオーガ,Pokémon,2
8,Maximum Belt,マキシマムベルト,Treinador,1


In [12]:
conexao.close()